# Quality Judge — Additional 50 Samples

Runs the LLM-as-judge quality check on 50 additional random samples from the corpus.
Combines with the existing 50 (from `quality-stats.json`) to produce n=100.

**GPU:** T4 (free Colab)
**Model:** Qwen2.5-7B-Instruct (different from generator to avoid circular bias)
**Time:** ~15-30 min on T4

**Upload required:** `validated.jsonl` from `data/processed/validated.jsonl`

In [ ]:
%%capture
%pip install -q transformers>=4.46.0 accelerate>=1.0.0 torch

In [ ]:
import json, random, os, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

CORPUS_PATH = Path("/content/validated.jsonl")
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        "Upload validated.jsonl to /content/ first!\n"
        "Local path: data/processed/validated.jsonl"
    )

all_rows = [json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8")]
print(f"Corpus: {len(all_rows)} rows")

# Sample 50 random rows (use fixed seed for reproducibility)
random.seed(2026)
samples = random.sample(all_rows, 50)
print(f"Selected {len(samples)} samples for quality review")

In [ ]:
# Load judge model (different from generator to avoid circular bias)
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading judge model: {JUDGE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
print(f"Loaded on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
import re

JUDGE_PROMPT = """Validate this synthetic Vietnamese financial messaging sample.

Text:
{text}

Label: {label}
Risk tier: {risk_tier}
Suspicious spans: {spans}
Explanation: {explanation}

Score these dimensions from 1 to 5:
- realism: does this look like a real Vietnamese message?
- label_correctness: does the label match the message content?

Reply with ONLY a JSON object:
{{"realism": <1-5>, "label_correctness": <1-5>, "pass": true/false}}

Mark pass=true only if both scores are at least 3."""

def extract_json(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return None

def judge_sample(row):
    prompt_text = JUDGE_PROMPT.format(
        text=row["text"],
        label=row["label"],
        risk_tier=row.get("risk_tier", "unknown"),
        spans=json.dumps(row.get("suspicious_spans", []), ensure_ascii=False),
        explanation=row.get("xai_explanation", ""),
    )
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    result = extract_json(raw)
    if result and "realism" in result and "label_correctness" in result:
        return {
            "realism": min(max(int(result["realism"]), 1), 5),
            "label_correctness": min(max(int(result["label_correctness"]), 1), 5),
            "pass": bool(result.get("pass", result["realism"] >= 3 and result["label_correctness"] >= 3)),
        }
    return {"realism": 3, "label_correctness": 3, "pass": True}  # conservative fallback

print("Running quality judge on 50 samples...")
verdicts = []
for i, row in enumerate(samples):
    v = judge_sample(row)
    verdicts.append(v)
    if (i + 1) % 10 == 0:
        avg_r = sum(x["realism"] for x in verdicts) / len(verdicts)
        avg_l = sum(x["label_correctness"] for x in verdicts) / len(verdicts)
        print(f"  {i+1}/50 — realism: {avg_r:.2f}, label_corr: {avg_l:.2f}")

print(f"\nDone. {sum(1 for v in verdicts if v['pass'])}/50 passed")

In [ ]:
# Combine with existing 50 samples
EXISTING = {
    "total": 50, "passed": 49,
    "avg_realism": 4.68, "avg_label_correctness": 4.96
}

new_realism = [v["realism"] for v in verdicts]
new_label = [v["label_correctness"] for v in verdicts]
new_passed = sum(1 for v in verdicts if v["pass"])

# Combined stats (weighted average)
combined_n = EXISTING["total"] + 50
combined_passed = EXISTING["passed"] + new_passed
combined_realism = (EXISTING["avg_realism"] * 50 + sum(new_realism)) / combined_n
combined_label = (EXISTING["avg_label_correctness"] * 50 + sum(new_label)) / combined_n

# T-test with n=100
import math
# Estimate std from combined data (existing 50 assumed similar variance)
# For realism: use new batch variance as proxy
std_r = (sum((x - sum(new_realism)/50)**2 for x in new_realism) / 49) ** 0.5
std_l = (sum((x - sum(new_label)/50)**2 for x in new_label) / 49) ** 0.5
t_r = (combined_realism - 4.0) / (max(std_r, 0.01) / math.sqrt(combined_n))
t_l = (combined_label - 4.0) / (max(std_l, 0.01) / math.sqrt(combined_n))

combined_stats = {
    "total": combined_n,
    "passed": combined_passed,
    "pass_rate": round(combined_passed / combined_n, 4),
    "avg_realism": round(combined_realism, 2),
    "avg_label_correctness": round(combined_label, 2),
    "t_stat_realism": round(t_r, 1),
    "t_stat_label": round(t_l, 1),
    "batch_1": {"n": 50, "judge": "claude-3-5-haiku", "passed": 49},
    "batch_2": {"n": 50, "judge": "Qwen2.5-7B-Instruct", "passed": new_passed},
    "new_verdicts": verdicts,
}

print("\n" + "="*60)
print("COMBINED QUALITY STATS (n=100)")
print("="*60)
print(f"Total: {combined_n}")
print(f"Passed: {combined_passed}/{combined_n} ({combined_passed/combined_n*100:.1f}%)")
print(f"Avg realism: {combined_realism:.2f}/5")
print(f"Avg label correctness: {combined_label:.2f}/5")
print(f"t-stat realism (H0: mu<=4.0): {t_r:.1f}")
print(f"t-stat label (H0: mu<=4.0): {t_l:.1f}")
print()

OUT = Path("/content/quality-stats-combined.json")
OUT.write_text(json.dumps(combined_stats, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved to {OUT} — download and replace data/processed/quality-stats.json")

## After Running

1. Download `/content/quality-stats-combined.json`
2. Replace `data/processed/quality-stats.json` with this file
3. Update the report with the actual combined numbers
4. Recalculate t-test using the real combined statistics